# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()
print(f"{metadata_dict['name']}: {metadata_dict['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} - {rs.get('name', '')}")
    # List first few fields in this record set
    if 'field' in rs:
        fields = rs['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields[:5]:
            if isinstance(field, dict):
                print(f"    - {field['@id']}: {field.get('name', field.get('description', ''))}")
            else:
                # field may be a string @id
                print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s found in the overview above.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Record Set @id list:", record_set_ids)

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} rows from record set {rs_id}.")

# Pick the first non-empty record set for demonstration
main_rs_id = next(iter(dataframes.keys()))
print(f"\nFields in {main_rs_id}:")
print(dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing numeric fields, and grouping data.

In [ ]:
# Select a numeric field (use one from columns in main_rs_id)
df = dataframes[main_rs_id]
# Try to auto-detect a suitable numeric column
numeric_field = None
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
        if df[col].dtype.kind in 'if' and df[col].notna().sum() > 0:
            numeric_field = col
            break
    except Exception:
        pass
if not numeric_field:
    raise ValueError("No suitable numeric field found.")

print(f"Using field '{numeric_field}' for numeric analysis.")

threshold = df[numeric_field].mean()  # use mean as threshold demo
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to pick a group field
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < 50:
        group_field = col
        break

if group_field:
    print(f"\nGrouping by '{group_field}'")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(grouped_df.head())
else:
    print("\nNo suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution of numeric field
plt.figure(figsize=(8,4))
df[numeric_field].hist(bins=30, alpha=0.7)
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.title(f'Distribution of {numeric_field} in {main_rs_id}')
plt.show()

# If group_field exists, plot mean per group
if group_field:
    group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
    group_means.plot(kind='bar', figsize=(10,5))
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to:
- Load and explore structured scientific metadata as defined by a Croissant JSON-LD schema
- Inspect available record sets and field `@id`s
- Load and analyze tabular data in context
- Apply basic filtering, normalization, and grouping operations
- Visualize numeric field distributions

This process can be extended for deeper domain analysis, feature engineering, or machine learning as needed. All references to the schema entities are always through their stable `@id`s, preserving interoperability and provenance for FAIR scientific computation.
